# Sprint 9: What could NYC's climate look like by 2050?

In Sprint 3 we found the NYC region warmed by about **+0.64 °F per decade** from 1981 to 2025, and hot days roughly **doubled**. This notebook asks the next question: **where is that heading?**

We'll answer it two ways and compare them:

| | Approach | What it assumes |
|---|---|---|
| **A** | Extend Sprint 3's straight line to 2050 | The future warms at exactly the same speed as the past |
| **B** | Use real **climate models** (CMIP6) | The physics of a world with more greenhouse gases |

**A few words you'll meet:**

- **Climate model:** a giant computer program that simulates the whole planet's air, oceans and ice, day by day, for decades. Scientists around the world run them for a big shared project called **CMIP6**.
- **Scenario:** a guess about how much greenhouse gas humans will release. Scientists name them like **SSP2-4.5** ("middle of the road") and **SSP5-8.5** ("very high emissions").
- **Ensemble:** a group of models. No single model is right, so we look at the **middle** and the **range** of several.

> ⚠️ **Important idea:** a climate model's "July 4, 2047" is *not* a forecast of that day's weather. Models get the **statistics** right (how warm a typical decade is, how many hot days a year), not the specific days. So we always compare **10-year averages**, never single days or single years.

**Where the model data comes from:** the free [Open-Meteo Climate API](https://open-meteo.com/en/docs/climate-api). It serves daily data from 7 high-resolution CMIP6 models for any point on Earth, 1950–2050, with **no API key**. These runs follow a **high-emissions path** (similar to SSP5-8.5). At the end there's an optional section that compares a middle and a high scenario.

## Step 1: Load the tools

Same as earlier sprints, plus two new ones.

- `from pathlib import Path`: a tool for building file paths (like `../data/file.csv`) that works on any computer.
- `import time`: lets us **pause** the program for a few seconds, so we don't flood the website with requests.
- `import numpy as np`: math tools. We use it for the straight-line fit, like in Sprint 3.
- `import pandas as pd`: tables (DataFrames).
- `import matplotlib.pyplot as plt`: charts.
- `import requests`: downloads data from websites, like in Sprint 1.

If you see `ModuleNotFoundError: No module named 'requests'`, run `pip install requests` in Terminal (with `(.venv)` on), then run this cell again.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

## Step 2: Settings we'll reuse

- `DATA_DIR` and `IMAGE_DIR`: where files go. The notebook runs inside `notebooks/`, so `..` means "go up one folder" to the project folder first.
- `LAT, LON`: the point we ask for. These are NYC's coordinates (west is negative). If your Sprint 1 notebook used slightly different numbers, use those instead so everything matches.
- `MODEL_DIR`: a folder where we'll save each model's download, so we only download once.
- `mkdir(exist_ok=True)`: makes the folder, and doesn't complain if it already exists.

In [ ]:
DATA_DIR = Path("../data")
IMAGE_DIR = Path("../docs/images")
MODEL_DIR = DATA_DIR / "climate_models"

LAT, LON = 40.71, -74.01

MODEL_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
print("Model files will be saved in:", MODEL_DIR.resolve())

## Step 3: Load our NASA history (1981–2025)

This is the same CSV the dashboard uses.

- `def c_to_f(celsius):` makes a small reusable function that turns °C into °F.
- `pd.read_csv(..., index_col=0, parse_dates=True)` reads the file, uses the first column (the date) as the row labels, and treats it as real dates.
- The next lines add columns: the high and the average in °F, plus the `year` and `month` of each row (we need the month later).
- `HOT_F = ...quantile(0.95)`: our "hot day" line, the hottest 5% of all highs. It's the same rule as the dashboard (about 81 °F).
- `obs_yearly = obs.groupby("year").agg(...)` makes **one row per year** with the average temperature and the number of hot days. "obs" is short for **observed**, meaning real history.

In [ ]:
def c_to_f(celsius):
    return celsius * 9 / 5 + 32


obs = pd.read_csv(DATA_DIR / "nyc_daily_1981_2025.csv", index_col=0, parse_dates=True)
obs["high_f"] = c_to_f(obs["T2M_MAX"])
obs["mean_f"] = c_to_f(obs["T2M"])
obs["year"] = obs.index.year
obs["month"] = obs.index.month

HOT_F = obs["high_f"].quantile(0.95)

obs_yearly = obs.groupby("year").agg(
    mean_f=("mean_f", "mean"),
    hot_days=("high_f", lambda highs: (highs >= HOT_F).sum()),
)

print(f"Hot day = a high of {HOT_F:.1f} °F or more")
obs_yearly.tail()

## Step 4 (Approach A): Just extend the straight line

This is Sprint 3's trend line, drawn longer.

- `np.polyfit(x, y, 1)` finds the best straight line through the points. The `1` means "a straight line" (a curve would be 2). It gives back the **slope** (°F per year) and the **intercept** (where the line starts).
- `np.arange(1981, 2051)` makes the list of years 1981, 1982, ... 2050. (The end number is left out, so we write 2051.)
- `trend_f = slope * all_years + intercept` is the line's value for every year: the school formula *y = m·x + b*.
- `slope * 10` turns "per year" into "per decade".

In [ ]:
slope, intercept = np.polyfit(obs_yearly.index, obs_yearly["mean_f"], 1)

all_years = np.arange(1981, 2051)
trend_f = pd.Series(slope * all_years + intercept, index=all_years)

print(f"Trend: {slope * 10:+.2f} °F per decade")
print(f"The straight line says 2050 will average {trend_f[2050]:.1f} °F")
print(f"That is {trend_f[2050] - trend_f[2025]:+.1f} °F warmer than the line's value for 2025")

Let's draw it. `plt.plot(x, y, ...)` draws a line; `"--"` makes it dashed; `label=` is the name shown in the legend. `plt.axvline(2025.5)` draws a vertical line where real data stops.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(obs_yearly.index, obs_yearly["mean_f"], color="black", label="NASA POWER (real)")
plt.plot(trend_f.index, trend_f, "--", color="tab:orange", label="Straight line, extended")
plt.axvline(2025.5, color="gray", linewidth=0.8)
plt.title("Approach A: if warming continues at the same speed")
plt.ylabel("Average temperature (°F)")
plt.legend()
plt.show()

### ✏️ Try this

1. Fit the line only to **2000–2025** instead of all years. Replace the first line of the Step 4 code with:
   ```python
   recent = obs_yearly.loc[2000:2025]
   slope, intercept = np.polyfit(recent.index, recent["mean_f"], 1)
   ```
   Does the 2050 number go up or down? Why do you think the start year matters so much?
2. Put the original line back before moving on (re-run the Step 4 cell).

**Why approach A isn't enough:** a straight line knows nothing about *why* it's warming. If emissions speed up, slow down or stop, the line can't know. Climate models can.

## Step 5 (Approach B): Download climate model data

These are the 7 CMIP6 models Open-Meteo offers. Each name is a model from a different research center (Italy, China, Taiwan, Japan, Europe, Germany, Japan again).

**The function `get_model_data(model)`** downloads one model:

- `url = ...` is the website address of the API.
- `params = {...}` is a **dictionary** (a list of name → value pairs) with our questions: which point, which dates, which model, which daily values.
  - `temperature_2m_mean` and `temperature_2m_max` are the daily average and daily high, 2 m above ground, in °C. They match NASA's `T2M` and `T2M_MAX`.
- `for attempt in range(3):` try up to 3 times. The free API has a speed limit; if we hit it, the site answers with code **429** ("too many requests").
- `if response.status_code == 429:` then wait 60 seconds (`time.sleep(60)`) and try again (`continue` jumps back to the top of the loop).
- `response.raise_for_status()` stops with an error if anything else went wrong. `break` means "it worked, leave the loop".
- After the loop, if we were *still* blocked 3 times, we stop with a clear message.
- `response.json()["daily"]` turns the answer into Python data and takes the `daily` part.
- `pd.DataFrame(daily)` makes it a table. The API sometimes adds the model name to the end of each column name (`temperature_2m_max_EC_Earth3P_HR`), so `.rename(...)` cuts that part off to keep names simple.
- We add a proper `date` column and a `model` column, so we can stack all models into one table later.

In [ ]:
MODELS = [
    "CMCC_CM2_VHR4",
    "FGOALS_f3_H",
    "HiRAM_SIT_HR",
    "MRI_AGCM3_2_S",
    "EC_Earth3P_HR",
    "MPI_ESM1_2_XR",
    "NICAM16_8S",
]


def get_model_data(model):
    url = "https://climate-api.open-meteo.com/v1/climate"
    params = {
        "latitude": LAT,
        "longitude": LON,
        "start_date": "1981-01-01",
        "end_date": "2050-12-31",
        "models": model,
        "daily": "temperature_2m_mean,temperature_2m_max",
    }
    for attempt in range(3):
        response = requests.get(url, params=params, timeout=180)
        if response.status_code == 429:
            print("   speed limit reached, waiting 60 seconds...")
            time.sleep(60)
            continue
        response.raise_for_status()
        break

    if response.status_code == 429:
        raise RuntimeError("still over the speed limit, try again in an hour")

    daily = response.json()["daily"]
    table = pd.DataFrame(daily)
    table = table.rename(columns=lambda name: name.replace("_" + model, ""))
    table["date"] = pd.to_datetime(table["time"])
    table["model"] = model
    return table.drop(columns="time")

**Now download every model, but only once.**

- `for model in MODELS:` repeats the indented lines for each model name.
- `file = MODEL_DIR / f"{model}.csv"` is where that model's data is saved.
- `if file.exists():` if we already downloaded it on an earlier run, just read the saved file. This makes re-running the notebook fast and kind to the free API.
- `else:` download it, save it with `.to_csv(...)`, then pause 5 seconds.
- `try: ... except Exception as error:` if one model fails, print why and **keep going** with the others, instead of stopping the whole notebook.
- `pieces.append(...)` adds each model's table to a list; `pd.concat(pieces)` stacks them into one big table.

⏳ The first run can take **5 to 10 minutes** (each model is 70 years of daily data). After that it takes a second. If a model is skipped, just run the cell again later: it will only fetch the missing ones.

In [ ]:
pieces = []
for model in MODELS:
    file = MODEL_DIR / f"{model}.csv"
    if file.exists():
        print("Already saved:", model)
        pieces.append(pd.read_csv(file, parse_dates=["date"]))
    else:
        print("Downloading:", model, "...")
        try:
            table = get_model_data(model)
            table.to_csv(file, index=False)
            pieces.append(table)
        except Exception as error:
            print("   skipped", model, "because:", error)
        time.sleep(5)

runs = pd.concat(pieces, ignore_index=True)
print()
print("Models we have:", runs["model"].nunique())
runs.head()

**Tidy it up**, the same way we did NASA's data:

- `dropna(...)` removes rows with missing values (a model may have a few gaps).
- Then we add °F columns, plus `year` and `month`. `.dt.year` means "take the year part of each date".
- `runs_yearly` is one row per year and **one column per model**. `.unstack("model")` is what turns the model names into columns.

In [ ]:
runs = runs.dropna(subset=["temperature_2m_mean", "temperature_2m_max"])
runs["mean_f"] = c_to_f(runs["temperature_2m_mean"])
runs["high_f"] = c_to_f(runs["temperature_2m_max"])
runs["year"] = runs["date"].dt.year
runs["month"] = runs["date"].dt.month

runs_yearly = runs.groupby(["year", "model"])["mean_f"].mean().unstack("model")
runs_yearly.tail()

## Step 6: Look at the raw model data next to the real history

Each thin colored line is one model. `for model in runs_yearly.columns:` draws them one by one. `alpha=0.6` makes them a bit see-through.

In [ ]:
plt.figure(figsize=(10, 4))
for model in runs_yearly.columns:
    plt.plot(runs_yearly.index, runs_yearly[model], linewidth=0.8, alpha=0.6, label=model)
plt.plot(obs_yearly.index, obs_yearly["mean_f"], color="black", linewidth=2, label="NASA POWER (real)")
plt.title("Raw climate model output vs real history")
plt.ylabel("Average temperature (°F)")
plt.legend(fontsize=7, ncol=2)
plt.show()

**Notice:** some models run warmer or colder than NASA's line the whole time, even in the past. This is called **bias**. A model might put NYC's grid box a bit too far inland, or its ocean a bit too cold.

The good news: scientists trust the **change** a model shows much more than its exact temperature. So we use the **change method** (scientists call it the "delta method"; *delta* means change):

1. For each model, find its average for **1981–2010** (the "baseline").
2. Find its average for **2041–2050**.
3. The difference is that model's **warming**.
4. Add that warming to **NASA's real 1981–2010 average**.

In code:

- `runs_yearly.loc[1981:2010].mean()` takes the rows for 1981 through 2010 and averages each column, so we get one number **per model**.
- `warming = future - baseline` does the subtraction for every model at once.
- `obs_base = ...` is NASA's real average for the same years.
- `.sort_values()` lists the models from least to most warming.

In [ ]:
model_baseline = runs_yearly.loc[1981:2010].mean()
model_future = runs_yearly.loc[2041:2050].mean()
warming = model_future - model_baseline

obs_base = obs_yearly.loc[1981:2010, "mean_f"].mean()

result = pd.DataFrame({
    "warming_f": warming,
    "projected_2040s_f": obs_base + warming,
}).sort_values("warming_f")

print(f"NASA's real 1981–2010 average: {obs_base:.1f} °F")
result.round(1)

**The ensemble answer.** `.median()` is the middle value (half the models are above it, half below). `.min()` and `.max()` show the range. Reporting a range is honest: it shows how much the models disagree.

In [ ]:
print(f"Warming by the 2040s vs 1981–2010: {warming.median():+.1f} °F "
      f"(models range from {warming.min():+.1f} to {warming.max():+.1f} °F)")
print(f"Projected 2041–2050 average: {obs_base + warming.median():.1f} °F")
print(f"Straight line (Approach A), average of 2041–2050: {trend_f.loc[2041:2050].mean():.1f} °F")
print(f"Real average of the last 10 years (2016–2025): {obs_yearly.loc[2016:2025, 'mean_f'].mean():.1f} °F")

### ✏️ Try this

1. Change the baseline years from `1981:2010` to `1991:2020` in **both** places in the cell above (models and NASA). Does the projected 2040s average change much? (It shouldn't change a lot. Why?)
2. Which model warms the most? Look it up online: which country's research center made it?

## Step 7: How many hot days could the 2040s have?

Hot days are trickier, because we need **daily** highs, not yearly averages. We fix the bias **month by month**, because a model can be too cold in winter but fine in summer.

- `obs_month_high` = NASA's average high for each month (Jan, Feb, ...) over 1981–2010. 12 numbers.
- `model_month_high` = each model's average high for each month, same years. 7 models × 12 months.
- `.reset_index()` turns it back into a normal table with `model`, `month` and `high_f` columns.
- `.map(obs_month_high)` looks up NASA's value for each row's month.
- `offset` = how much to add to that model, in that month, to line it up with NASA.
- `runs.merge(...)` attaches the right offset to every single day, matching on model **and** month.
- `high_adj_f` = the model's daily high plus its offset. "adj" means **adjusted**.

In [ ]:
obs_base_days = obs[(obs["year"] >= 1981) & (obs["year"] <= 2010)]
runs_base_days = runs[(runs["year"] >= 1981) & (runs["year"] <= 2010)]

obs_month_high = obs_base_days.groupby("month")["high_f"].mean()
model_month_high = runs_base_days.groupby(["model", "month"])["high_f"].mean().reset_index()

model_month_high["obs_high_f"] = model_month_high["month"].map(obs_month_high)
model_month_high["offset"] = model_month_high["obs_high_f"] - model_month_high["high_f"]

runs = runs.drop(columns="offset", errors="ignore")
runs = runs.merge(model_month_high[["model", "month", "offset"]], on=["model", "month"])
runs["high_adj_f"] = runs["high_f"] + runs["offset"]

model_month_high.head(12).round(1)

(`errors="ignore"` on the `drop` line just means: if you run this cell twice, remove the old `offset` column first instead of making a duplicate.)

**Count hot days per year, per model.** `lambda highs: (highs >= HOT_F).sum()` is a tiny unnamed function: it checks every day's high against our hot-day line and counts the `True`s, exactly like the dashboard does.

In [ ]:
runs_hot = (
    runs.groupby(["year", "model"])["high_adj_f"]
    .apply(lambda highs: (highs >= HOT_F).sum())
    .unstack("model")
)

hot_base = obs_yearly.loc[1981:2010, "hot_days"].mean()
hot_recent = obs_yearly.loc[2016:2025, "hot_days"].mean()
hot_future = runs_hot.loc[2041:2050].mean()

print(f"Real hot days a year, 1981–2010: {hot_base:.0f}")
print(f"Real hot days a year, 2016–2025: {hot_recent:.0f}")
print(f"Models, 2041–2050: {hot_future.median():.0f} a year "
      f"(range {hot_future.min():.0f} to {hot_future.max():.0f})")

### ✏️ Try this

1. Change the hot-day line to a fixed **90 °F**: add `HOT_F = 90` at the top of the cell above and run it. How many 90 °F days do the models expect in the 2040s? (Remember: gridded data runs a few degrees cooler than Central Park.)
2. Run Step 3 again afterwards to put `HOT_F` back to normal.

## Step 8: The big picture chart

Single model years jump around a lot (that's weather), so we smooth each line with a **10-year rolling average**: `.rolling(10).mean()` means "for each year, average it with the 9 years before it". So the value at 2050 is the 2041–2050 average.

- `adjusted_yearly` applies the change method to every year: model value − model's own 1981–2010 average + NASA's 1981–2010 average.
- `ens["median"]`, `ens["low"]`, `ens["high"]` are the middle and the lowest/highest model each year (`axis=1` means "across the columns", i.e. across models).
- `plt.fill_between(...)` shades the area between the lowest and highest model.
- `plt.savefig(...)` saves the picture for the README. It must come **before** `plt.show()`.

In [ ]:
adjusted_yearly = runs_yearly - model_baseline + obs_base
adjusted_smooth = adjusted_yearly.rolling(10).mean()

ens = pd.DataFrame({
    "median": adjusted_smooth.median(axis=1),
    "low": adjusted_smooth.min(axis=1),
    "high": adjusted_smooth.max(axis=1),
})
obs_smooth = obs_yearly["mean_f"].rolling(10).mean()

plt.figure(figsize=(10, 5))
plt.fill_between(ens.index, ens["low"], ens["high"], color="tab:red", alpha=0.2, label="Range of climate models")
plt.plot(ens.index, ens["median"], color="tab:red", linewidth=2, label="Climate models (middle)")
plt.plot(trend_f.index, trend_f.rolling(10).mean(), "--", color="tab:orange", label="Straight line, extended")
plt.plot(obs_smooth.index, obs_smooth, color="black", linewidth=2.5, label="NASA POWER (real)")
plt.axvline(2025.5, color="gray", linewidth=0.8)
plt.title("NYC region: 10-year average temperature, 1990–2050")
plt.ylabel("°F (10-year rolling average)")
plt.xlim(1990, 2050)
plt.legend(loc="upper left")
plt.savefig(IMAGE_DIR / "future_2050.png", dpi=150, bbox_inches="tight")
plt.show()

**How to read it:** where the black line (real) and the red line (models) overlap in the past, the models are doing a good job. After 2025 the red band shows where we might be heading. If the orange dashed line sits **below** the red line, the models expect warming to **speed up** compared to the past 45 years.

## Step 9: Save the results for the dashboard

The dashboard shouldn't download 7 models every time someone opens it, so we save two small files:

1. `future_2050_yearly.csv`: one row per year (1990–2050) with the lines from the chart, plus hot days.
2. `future_2050_summary.csv`: the headline numbers, one row.

`pd.DataFrame({...})` builds a table from named columns. `.loc[1990:2050]` keeps only those years. `.round(2)` keeps 2 decimals so the file stays small and readable.

In [ ]:
hot_smooth = runs_hot.rolling(10).mean()

yearly_out = pd.DataFrame({
    "observed_f": obs_smooth,
    "models_median_f": ens["median"],
    "models_low_f": ens["low"],
    "models_high_f": ens["high"],
    "straight_line_f": trend_f.rolling(10).mean(),
    "observed_hot_days": obs_yearly["hot_days"].rolling(10).mean(),
    "models_hot_days_median": hot_smooth.median(axis=1),
    "models_hot_days_low": hot_smooth.min(axis=1),
    "models_hot_days_high": hot_smooth.max(axis=1),
}).loc[1990:2050].round(2)
yearly_out.index.name = "year"
yearly_out.to_csv(DATA_DIR / "future_2050_yearly.csv")

summary = pd.DataFrame([{
    "hot_threshold_f": HOT_F,
    "n_models": runs_yearly.shape[1],
    "base_mean_f": obs_base,
    "recent_mean_f": obs_yearly.loc[2016:2025, "mean_f"].mean(),
    "future_mean_median_f": obs_base + warming.median(),
    "future_mean_low_f": obs_base + warming.min(),
    "future_mean_high_f": obs_base + warming.max(),
    "straight_line_2040s_f": trend_f.loc[2041:2050].mean(),
    "base_hot_days": hot_base,
    "recent_hot_days": hot_recent,
    "future_hot_days_median": hot_future.median(),
    "future_hot_days_low": hot_future.min(),
    "future_hot_days_high": hot_future.max(),
}]).round(2)
summary.to_csv(DATA_DIR / "future_2050_summary.csv", index=False)

summary.T

## Step 10: What about a lower-emissions future?

The 7 models above all follow a **high-emissions** path. What if the world cuts emissions? The IPCC (the UN's climate science panel) summarized it for the whole planet, compared with the late 1800s:

| Scenario | 2041–2060 | 2081–2100 |
|---|---|---|
| SSP2-4.5 (middle of the road) | about +2.0 °C | about +2.7 °C |
| SSP5-8.5 (very high emissions) | about +2.4 °C | about +4.4 °C |

*Source: IPCC AR6 Working Group I, Summary for Policymakers, Table SPM.1 (best estimates).*

**The surprise:** by 2050 the two paths are only about **0.4 °C (0.7 °F)** apart worldwide. Much of the warming until 2050 is already "baked in" by gases we've released. The choices we make now mostly decide the **second half** of the century, where the gap grows to almost 2 °C.

So our 2040s numbers are a fair picture of the next 25 years under either path, perhaps a little on the warm side.

### 🧗 Optional stretch: compare two scenarios yourself

NASA publishes **NEX-GDDP-CMIP6**: CMIP6 models already downscaled to a ~25 km grid, for both SSP2-4.5 and SSP5-8.5. We can ask NASA's server for just one grid point, one year at a time.

This section is **switched off** (`RUN_SCENARIOS = False`), so the notebook works without it. To try it, change it to `True` and run the cells. It makes about 40 small downloads (a few minutes).

- `NEX_LAT, NEX_LON` is **Central Park**. This dataset only covers land, and the NYC point we used above sits close to the water, so we pick a spot that's clearly on land.
- `LON % 360`: this dataset counts longitude from 0 to 360 instead of −180 to 180. `%` is "remainder after dividing", which turns −73.97 into 286.03.
- NASA sometimes re-released a year's file with a version tag like `_v1.1`, so the function tries the plain name first, then the tagged ones.
- The values come in **Kelvin** (scientists' temperature scale, where 0 is the coldest possible). `- 273.15` turns Kelvin into °C, then we use our `c_to_f`.
- `io.StringIO(text)` lets pandas read text we already downloaded as if it were a file.

In [ ]:
RUN_SCENARIOS = False

import io

NEX_MODEL = "ACCESS-CM2"          # one well-known model from Australia
NEX_LAT, NEX_LON = 40.78, -73.97  # Central Park
NEX_FILE = DATA_DIR / f"nex_gddp_{NEX_MODEL}.csv"


def get_nex_year(scenario, year):
    base = "https://ds.nccs.nasa.gov/thredds/ncss/grid/AMES/NEX/GDDP-CMIP6"
    for version in ["", "_v1.1", "_v1.2"]:
        name = f"tasmax_day_{NEX_MODEL}_{scenario}_r1i1p1f1_gn_{year}{version}.nc"
        url = f"{base}/{NEX_MODEL}/{scenario}/r1i1p1f1/tasmax/{name}"
        params = {
            "var": "tasmax",
            "latitude": NEX_LAT,
            "longitude": NEX_LON % 360,
            "time_start": f"{year}-01-01T00:00:00Z",
            "time_end": f"{year}-12-31T23:59:59Z",
            "accept": "csv",
        }
        response = requests.get(url, params=params, timeout=120)
        if response.status_code == 200:
            table = pd.read_csv(io.StringIO(response.text))
            time_column = [c for c in table.columns if c.startswith("time")][0]
            value_column = table.columns[-1]
            return pd.DataFrame({
                "date": pd.to_datetime(table[time_column].str[:10]),
                "high_f": c_to_f(table[value_column] - 273.15),
                "scenario": scenario,
            })
    raise RuntimeError(f"NASA had no file for {scenario} {year}. Last address tried: {response.url}")


if RUN_SCENARIOS and not NEX_FILE.exists():
    wanted = [("historical", y) for y in range(1995, 2015)]
    wanted += [("ssp245", y) for y in range(2041, 2051)]
    wanted += [("ssp585", y) for y in range(2041, 2051)]
    pieces = []
    for scenario, year in wanted:
        print("Downloading", scenario, year)
        pieces.append(get_nex_year(scenario, year))
        time.sleep(1)
    pd.concat(pieces).to_csv(NEX_FILE, index=False)

if NEX_FILE.exists():
    nex = pd.read_csv(NEX_FILE, parse_dates=["date"])
    print(nex.groupby("scenario")["high_f"].describe().round(1))
else:
    print("Stretch section is switched off. Set RUN_SCENARIOS = True to try it.")

**Compare the scenarios** with the same two tricks as before: the change method for averages, and the month-by-month offset for hot days. Here the baseline is **1995–2014**, the standard years the IPCC uses for "recent past".

- `hist` is the model's past; `obs_ref` is NASA's real data for the same 20 years.
- `offset` lines the model up with NASA, month by month.
- For each scenario, we print how much warmer the average daily high is, and the hot days per year.

In [ ]:
if NEX_FILE.exists():
    nex["month"] = nex["date"].dt.month
    hist = nex[nex["scenario"] == "historical"]
    obs_ref = obs[(obs["year"] >= 1995) & (obs["year"] <= 2014)]

    offset = obs_ref.groupby("month")["high_f"].mean() - hist.groupby("month")["high_f"].mean()
    nex["high_adj_f"] = nex["high_f"] + nex["month"].map(offset)

    real_hot = (obs_ref["high_f"] >= HOT_F).sum() / 20
    print(f"Real, 1995–2014: {real_hot:.0f} hot days a year")
    for scenario in ["ssp245", "ssp585"]:
        future = nex[nex["scenario"] == scenario]
        change = future["high_f"].mean() - hist["high_f"].mean()
        hot = (future["high_adj_f"] >= HOT_F).sum() / 10
        print(f"{scenario}, 2041–2050: highs {change:+.1f} °F warmer, {hot:.0f} hot days a year")

If the two scenarios look **close** in the 2040s, that matches the IPCC table: the paths split mostly after 2050.

> If NASA's server says it has no file, it may have renamed things since this was written. That's fine: this part is optional, and the rest of the notebook doesn't need it.

---

## What we learned

- Extending a straight line is a quick first guess, but it can't know if warming speeds up.
- Climate models disagree on the exact temperature (bias), so we trust their **change**, and add it to real history.
- We report the **middle and the range** of several models, not one number.
- Up to 2050, the emissions scenarios are fairly close. Our choices mostly shape what happens after.

## Limits

- The models' grid boxes (about 20–50 km) describe the **NYC region**, like NASA POWER. Dense neighborhoods run hotter.
- Our month-by-month fix shifts the average but doesn't fix how *spread out* a model's days are, so hot-day counts are rougher than average temperatures.
- 7 models is a small ensemble. The IPCC uses dozens.

### ✏️ Final try this

Write 2–3 sentences, in your own words, answering: *"How much hotter could NYC be by the 2040s, and how sure are we?"* Use the numbers you printed in Steps 6 and 7. You'll paste them into `docs/findings.md`.